### Silver - Vendedores

Problemas identificados:
- `V004` duplicado: uma versão com canal `CH02` (existente) e outra com
  `CH99` (não existente na dimensão de canal). Mantida a versão com
  `CH02`
- `V008` duplicado, com a segunda versão apresentando "duplicado" no
  nome
- `canal_id` com `ch07` em caixa baixa
- `regional_code` combinando sigla (`S`, `N`, `NE`...) com nome por
  extenso (`sul`)
- `hire_date` em 2 formatos (`yyyy-MM-dd` e `dd/MM/yyyy`)
- `status` com `Ativo`/`ativo`/`inativo`/vazio

Valores nulos em `canal_id` e `regional_code` são preservados como nulo,
vendedor sem canal ou região definida é uma condição de negócio válida,
não uma falha de formatação.

In [ ]:
%run "../00_setup/00_setup_ambiente"

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_vendedores_bronze = spark.table(f"{schema_name}.bronze_vendedores")
display(df_vendedores_bronze)

In [ ]:
# Deduplicação mantendo a primeira ocorrência: resolve os dois casos
# (V004 permanece com CH02, a primeira linha; V008 permanece sem o sufixo
# "duplicado" no nome)
w_dedup = Window.partitionBy("seller_id").orderBy(F.monotonically_increasing_id())

df_vendedores_dedup = (
    df_vendedores_bronze
    .withColumn("rk", F.row_number().over(w_dedup))
    .filter(F.col("rk") == 1)
    .drop("rk")
)

In [ ]:
df_vendedores_norm = (
    df_vendedores_dedup
    .withColumn("canal_id", F.upper(F.trim(F.col("canal_id"))))
    .withColumn(
        "regional_code_norm",
        F.when(F.lower(F.trim(F.col("regional_code"))) == "sul", F.lit("S"))
         .otherwise(F.upper(F.trim(F.col("regional_code"))))
    )
)

Utilização de try_to_date em vez de to_date: o runtime do Databricks
opera em modo ANSI, no qual to_date lança exceção quando o valor não
corresponde ao formato especificado, em vez de retornar null. Esse
comportamento interrompia o coalesce na primeira tentativa de formato
incompatível. try_to_date retorna null nesses casos, permitindo que o
coalesce avalie a próxima tentativa.

In [ ]:
df_vendedores_silver = (
    df_vendedores_norm
    .withColumn(
        "hire_date_parsed",
        F.coalesce(
            F.expr("try_to_date(hire_date, 'yyyy-MM-dd')"),
            F.expr("try_to_date(hire_date, 'dd/MM/yyyy')"),
        )
    )
    .withColumn(
        "status_vendedor",
        F.when(F.lower(F.trim(F.col("status"))) == "ativo", "Ativo")
         .when(F.lower(F.trim(F.col("status"))) == "inativo", "Inativo")
         .otherwise("Não informado")
    )
    .select(
        F.col("seller_id"),
        F.col("seller_name").alias("nome_vendedor"),
        F.col("canal_id"),
        F.col("regional_code_norm").alias("regional_code"),
        F.col("hire_date_parsed").alias("data_contratacao"),
        "status_vendedor",
    )
)

(
    df_vendedores_silver.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{schema_name}.silver_vendedores")
)

print("bronze:", df_vendedores_bronze.count(), "-> silver:", df_vendedores_silver.count())
display(df_vendedores_silver)